# reshape, permutation et canal : donner les axes attendus

Suivre une case pour distinguer un redécoupage, un échange et un axe ajouté.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#organiser-les-axes).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Le modèle peut refuser une entrée dont les valeurs sont bonnes, mais les axes mal organisés. Pour voir les déplacements sans confondre les cases, prenons une expérience indépendante : les identifiants 1 à 6, dans deux lignes de trois. Ce ne sont pas des mesures audio.

Les deux sorties ont la forme (3,2), mais pas les mêmes lignes. reshape reprend la suite 1,2,3,4,5,6 et la coupe autrement. transpose échange les deux axes : l’ancienne colonne [1,4] devient la première ligne. Une shape correcte ne prouve donc pas que les valeurs sont au bon endroit.

Passons au besoin d’une couche audio : elle peut attendre (extraits,canaux,temps), alors qu’un tableau mono ne possède que (extraits,temps). Ajoutons l’axe manquant, sans modifier l’ordre des nombres.

unsqueeze(1) donne (2,1,3). L’écriture source[:,None,:] est équivalente. permute(0,2,1) demande ensuite l’ancien axe 0, puis 2, puis 1 : on obtient (2,3,1). Sur une entrée (batch,temps,fréquences), ce même ordre donnerait (batch,fréquences,temps). reshape ne ferait pas cet échange sémantique.

Imaginons maintenant le dernier lot, qui ne contient qu’un extrait : (1,1,3). Nous voulons retirer son canal, tout en gardant l’axe qui compte les extraits.

| Expression | Forme | Ce qui reste |
| --- | --- | --- |
| squeeze(1) | (1,3) | Un extrait, trois positions |
| squeeze() | (3,) | L’axe des extraits a aussi disparu |

La différence reste invisible si l’on regarde seulement les valeurs.

Après une transposition, view(-1) peut également échouer parce que la disposition mémoire n’est plus compatible. transpose.reshape(-1) suit ici [1,4,2,5,3,6] et accepte une copie si nécessaire. Ce point ne change pas la distinction entre redécouper les valeurs et permuter leurs axes.

## Démonstration 1 · Retrouver les cases et retirer le bon axe

Retrouver les objets et exécuter les fragments du chapitre dans leur contexte complet.

### Préparer les bibliothèques

Les nombres identifient des cases ; les deux opérations ont une forme identique mais des valeurs différentes.

In [3]:
import torch


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [4]:
source = torch.arange(1, 7).reshape(2, 3)
redecoupe = source.reshape(3, 2)
transpose = source.transpose(0, 1)
aplati = transpose.reshape(-1)


In [5]:
observer('source', source)
observer('redecoupe', redecoupe)
observer('transpose', transpose)
observer('aplati', aplati)

source forme (2, 3) type torch.int64 requires_grad False
  valeurs : [[1, 2, 3], [4, 5, 6]]
redecoupe forme (3, 2) type torch.int64 requires_grad False
  valeurs : [[1, 2], [3, 4], [5, 6]]
transpose forme (3, 2) type torch.int64 requires_grad False
  valeurs : [[1, 4], [2, 5], [3, 6]]
aplati forme (6,) type torch.int64 requires_grad False
  valeurs : [1, 4, 2, 5, 3, 6]


### Calculer l’état suivant

La nouvelle taille un représente un axe, pas une duplication.

In [6]:
avec_canal = source.unsqueeze(1)
autre_ecriture = source[:, None, :]


In [7]:
observer('avec_canal', avec_canal)
observer('autre_ecriture', autre_ecriture)

avec_canal forme (2, 1, 3) type torch.int64 requires_grad False
  valeurs : [[[1, 2, 3]], [[4, 5, 6]]]
autre_ecriture forme (2, 1, 3) type torch.int64 requires_grad False
  valeurs : [[[1, 2, 3]], [[4, 5, 6]]]


### Calculer l’état suivant

La liste 0,2,1 place l’ancien temps avant l’ancien canal.

In [8]:
temps_avant = avec_canal.permute(0, 2, 1)


In [9]:
observer('temps_avant', temps_avant)

temps_avant forme (2, 3, 1) type torch.int64 requires_grad False
  valeurs : [[[1], [2], [3]], [[4], [5], [6]]]


### Calculer l’état suivant

Sur un lot de taille un, squeeze sans argument retire aussi le batch.

In [10]:
dernier_lot = avec_canal[1:]
retrait_cible = dernier_lot.squeeze(1)
retrait_total = dernier_lot.squeeze()


In [11]:
observer('dernier_lot', dernier_lot)
observer('retrait_cible', retrait_cible)
observer('retrait_total', retrait_total)

dernier_lot forme (1, 1, 3) type torch.int64 requires_grad False
  valeurs : [[[4, 5, 6]]]
retrait_cible forme (1, 3) type torch.int64 requires_grad False
  valeurs : [[4, 5, 6]]
retrait_total forme (3,) type torch.int64 requires_grad False
  valeurs : [4, 5, 6]


### Lire les sorties

Les sorties comparent les valeurs et les formes, sans modifier la source.

In [12]:
print("reshape :", redecoupe.tolist())
print("transpose :", transpose.tolist())
print("ordre logique aplati :", aplati.tolist())
print("canal :", tuple(avec_canal.shape))


reshape : [[1, 2], [3, 4], [5, 6]]
transpose : [[1, 4], [2, 5], [3, 6]]
ordre logique aplati : [1, 4, 2, 5, 3, 6]
canal : (2, 1, 3)


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [13]:
print("None equivalent :", torch.equal(avec_canal, autre_ecriture))
print("axes permutes :", tuple(temps_avant.shape))
print("dernier lot :", tuple(dernier_lot.shape))
print("squeeze(1) :", tuple(retrait_cible.shape))


None equivalent : True
axes permutes : (2, 3, 1)
dernier lot : (1, 1, 3)
squeeze(1) : (1, 3)


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [14]:
print("squeeze() :", tuple(retrait_total.shape))


squeeze() : (3,)


In [15]:
assert bool(redecoupe.tolist() == [[1,2],[3,4],[5,6]] and transpose.tolist() == [[1,4],[2,5],[3,6]]), 'organiser-les-axes/main: redecoupe.tolist() == [[1,2],[3,4],[5,6]] and transpose.tolist() == [[1,4],[2,5],[3,6]]'
assert bool(aplati.tolist() == [1,4,2,5,3,6] and not transpose.is_contiguous()), 'organiser-les-axes/main: aplati.tolist() == [1,4,2,5,3,6] and not transpose.is_contiguous()'
assert bool(tuple(avec_canal.shape) == (2,1,3) and torch.equal(avec_canal, autre_ecriture)), 'organiser-les-axes/main: tuple(avec_canal.shape) == (2,1,3) and torch.equal(avec_canal, autre_ecriture)'
assert bool(tuple(temps_avant.shape) == (2,3,1) and tuple(retrait_cible.shape) == (1,3) and tuple(retrait_total.shape) == (3,)), 'organiser-les-axes/main: tuple(temps_avant.shape) == (2,3,1) and tuple(retrait_cible.shape) == (1,3) and tuple(retrait_total.shape) == (3,)'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Avec la grille [[1,2,3],[4,5,6]], comparez la case [1,0] après reshape(3,2), puis après transpose(0,1). Donnez les deux valeurs dans cet ordre.

In [16]:
exo_x = torch.arange(1,7).reshape(2,3)
exo_r = exo_x.reshape(3,2)
exo_t = exo_x.transpose(0,1)
exo_valeurs = ...  # Remplacer les points par votre réponse.
if exo_valeurs is Ellipsis or exo_valeurs is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_valeurs) == (exo_r[1,0].item(),exo_t[1,0].item())
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

reshape conserve le parcours 1 à 6 ; transpose échange le rôle des axes. Une forme identique ne garantit pas le même contenu.

</details>

### Exercice 2

Ajoutez un axe canal en position 1 à cette grille 2×3, sans répéter ses valeurs.

In [17]:
exo_x = torch.arange(1,7).reshape(2,3)
exo_avec_canal = ...  # Remplacer les points par votre réponse.
if exo_avec_canal is Ellipsis or exo_avec_canal is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_avec_canal.shape) == (2,1,3)
    assert torch.equal(exo_avec_canal.squeeze(1),exo_x)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

unsqueeze insère un axe de taille un ; squeeze(1) vérifie ensuite la conservation des valeurs.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_x = torch.arange(1,7).reshape(2,3)
exo_r = exo_x.reshape(3,2)
exo_t = exo_x.transpose(0,1)
exo_valeurs = (3,2)
if exo_valeurs is Ellipsis or exo_valeurs is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_valeurs) == (exo_r[1,0].item(),exo_t[1,0].item())
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_x = torch.arange(1,7).reshape(2,3)
exo_avec_canal = exo_x.unsqueeze(1)
if exo_avec_canal is Ellipsis or exo_avec_canal is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_avec_canal.shape) == (2,1,3)
    assert torch.equal(exo_avec_canal.squeeze(1),exo_x)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [20]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()